# LAB 03 — Naive Bayes: từ bằng chứng đến xác suất

**DNU Learning Analytics Lab**

- **Họ tên:** ...
- **MSSV:** ...
- **Lớp:** ...

> **Câu hỏi trung tâm:** Nếu K-NN dự đoán bằng những sinh viên ‘gần’ nhất, Naive Bayes sẽ dự đoán cùng một sinh viên như thế nào bằng xác suất?

Lab 03 dùng **cùng bài toán, cùng feature, cùng target và cùng train/test split của Lab 02** để việc so sánh có ý nghĩa.

In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
from nb_manual import compute_class_priors, gaussian_log_likelihood
DATA_PATH = Path('data/StudentPerformanceFactors.csv')
print('Environment ready! Dataset exists:', DATA_PATH.exists())

# Mission 1 — Back to DNU case study

Tạo nhãn giảng dạy `Needs_Support = 1` nếu `Exam_Score < 65`, ngược lại là 0. Giữ đúng bài toán của Lab 02.

**Quan trọng:** không dùng `Exam_Score` làm feature vì sẽ gây target leakage.

**[TRẢ LỜI]** Vì sao đây là leakage? Vì sao cần cùng train/test split như Lab 02?

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError('Run: python scripts/download_data.py')
df = pd.read_csv(DATA_PATH).copy()
df['Needs_Support'] = (df['Exam_Score'] < 65).astype(int)
feature_cols = ['Hours_Studied', 'Attendance', 'Previous_Scores', 'Sleep_Hours']
assert 'Exam_Score' not in feature_cols
model_df = df[feature_cols + ['Needs_Support']].dropna().copy()
X = model_df[feature_cols]
y = model_df['Needs_Support']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print('Train:', X_train.shape, 'Test:', X_test.shape)
display(y_train.value_counts(normalize=True).rename('prior').to_frame())

# Mission 2 — Prior first

Hoàn thiện `compute_class_priors(y)` trong `nb_manual.py`.

**[TRẢ LỜI]** `P(Needs_Support=1)` có ý nghĩa gì? Prior có phụ thuộc vào feature của sinh viên mới không?

In [ ]:
try:
    priors_manual = compute_class_priors(y_train.to_numpy())
    print('Manual priors:', priors_manual)
    print('Pandas check:', y_train.value_counts(normalize=True).sort_index().to_dict())
except NotImplementedError as e:
    print('TODO in nb_manual.py ->', e)

# Mission 3 — Gaussian evidence: model học gì?

Hoàn thiện `gaussian_log_likelihood(x, mean, var)` trong `nb_manual.py`, sau đó huấn luyện `GaussianNB`.

Quan sát `class_prior_`, `theta_`, `var_`.

**[TRẢ LỜI]** `theta_` và `var_` biểu diễn gì? GaussianNB có lưu toàn bộ hàng xóm như K-NN không?

In [ ]:
gnb = GaussianNB()
gnb.fit(X_train, y_train)
print('Classes:', gnb.classes_)
print('Class priors:', gnb.class_prior_)
theta_df = pd.DataFrame(gnb.theta_, index=[f'class_{c}' for c in gnb.classes_], columns=feature_cols)
var_df = pd.DataFrame(gnb.var_, index=[f'class_{c}' for c in gnb.classes_], columns=feature_cols)
display(theta_df)
display(var_df)
sample0 = X_test.iloc[0]
attendance_idx = feature_cols.index('Attendance')
x_value = float(sample0['Attendance'])
for row_idx, class_label in enumerate(gnb.classes_):
    try:
        ll = gaussian_log_likelihood(x_value, float(gnb.theta_[row_idx, attendance_idx]), float(gnb.var_[row_idx, attendance_idx]))
        print(f'class={class_label}, Attendance log-likelihood={ll:.4f}')
    except NotImplementedError as e:
        print('TODO in nb_manual.py ->', e)
        break

# Mission 4 — Evaluate beyond Accuracy

Đánh giá Accuracy, Precision, Recall, F1 và Confusion Matrix.

Đặc biệt chú ý **False Negative**: sinh viên thực tế cần hỗ trợ (`1`) nhưng model dự đoán không cần hỗ trợ (`0`).

**[TRẢ LỜI]** Vì sao chỉ nhìn Accuracy là chưa đủ?

In [ ]:
def evaluate_binary(name, y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return {'model': name, 'accuracy': accuracy_score(y_true,y_pred), 'precision': precision_score(y_true,y_pred,zero_division=0), 'recall': recall_score(y_true,y_pred,zero_division=0), 'f1': f1_score(y_true,y_pred,zero_division=0), 'tn':tn, 'fp':fp, 'fn':fn, 'tp':tp}
nb_pred = gnb.predict(X_test)
nb_result = evaluate_binary('GaussianNB', y_test, nb_pred)
display(pd.DataFrame([nb_result]))
ConfusionMatrixDisplay.from_predictions(y_test, nb_pred, display_labels=['No Support','Needs Support'])
plt.title('GaussianNB — Confusion Matrix')
plt.show()
print(classification_report(y_test, nb_pred, target_names=['No Support','Needs Support'], zero_division=0))

# Mission 5 — Explain one student

Chọn một sample trong `X_test`, xem `predict_proba()` và thử tính log-score từ log prior + Gaussian log-likelihood của từng feature.

> Đây không phải feature importance; ta đang xem likelihood cho một dự đoán cụ thể.

**[TRẢ LỜI]** Model dự đoán lớp nào? Bằng chứng nào có vẻ nghiêng nhiều hơn về lớp đó?

In [ ]:
student_x = X_test.iloc[[0]]
display(student_x)
pred_class = int(gnb.predict(student_x)[0])
proba = gnb.predict_proba(student_x)[0]
for cls, p in zip(gnb.classes_, proba):
    print(f'P(class={cls} | X) = {p:.6f}')
try:
    x_arr = student_x.iloc[0].to_numpy(dtype=float)
    rows = []
    for i, cls in enumerate(gnb.classes_):
        log_prior = math.log(float(gnb.class_prior_[i]))
        feature_ll = [gaussian_log_likelihood(float(x_arr[j]), float(gnb.theta_[i,j]), float(gnb.var_[i,j])) for j in range(len(feature_cols))]
        rows.append({'class': int(cls), 'log_prior': log_prior, **{f'll_{f}':v for f,v in zip(feature_cols,feature_ll)}, 'total_log_score': log_prior + sum(feature_ll)})
    score_df = pd.DataFrame(rows)
    display(score_df)
    print('Manual argmax:', int(score_df.loc[score_df.total_log_score.idxmax(),'class']), 'sklearn:', pred_class)
except NotImplementedError as e:
    print('TODO in nb_manual.py ->', e)

# Mission 6 — K-NN vs Naive Bayes

So sánh hai cách suy luận trên **cùng test set**. K-NN tham chiếu dùng `K=5`, Euclidean và `StandardScaler`.

**[TRẢ LỜI]** Tìm ít nhất 3 sample hai model dự đoán khác nhau và giải thích vì sao chúng có thể bất đồng.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
knn = KNeighborsClassifier(n_neighbors=5, metric='euclidean')
knn.fit(X_train_scaled, y_train)
knn_pred = knn.predict(X_test_scaled)
knn_result = evaluate_binary('KNN k=5 scaled', y_test, knn_pred)
display(pd.DataFrame([nb_result, knn_result])[['model','accuracy','precision','recall','f1','fn','fp']])
comparison = X_test.reset_index(drop=True).copy()
comparison['Actual'] = y_test.reset_index(drop=True)
comparison['NB_Pred'] = nb_pred
comparison['KNN_Pred'] = knn_pred
comparison['Same'] = comparison['NB_Pred'] == comparison['KNN_Pred']
disagreements = comparison[~comparison['Same']].copy()
print('Number of disagreements:', len(disagreements))
display(disagreements.head(10))

# Transfer Challenge — Naive Bayes không chỉ dành cho Student Performance

Pipeline: `Text → CountVectorizer → word counts → MultinomialNB → Topic`.

**[TRẢ LỜI]** Vì sao văn bản đếm từ dùng MultinomialNB thay vì GaussianNB?

Mặc định `RUN_TRANSFER=False` để GitHub Actions không phụ thuộc vào tải 20 Newsgroups.

In [ ]:
RUN_TRANSFER = False
if RUN_TRANSFER:
    from sklearn.datasets import fetch_20newsgroups
    categories = ['comp.graphics','rec.sport.baseball','sci.space','talk.politics.misc']
    train_text = fetch_20newsgroups(subset='train', categories=categories, remove=('headers','footers','quotes'))
    test_text = fetch_20newsgroups(subset='test', categories=categories, remove=('headers','footers','quotes'))
    vectorizer = CountVectorizer(stop_words='english', max_features=10000)
    X_text_train = vectorizer.fit_transform(train_text.data)
    X_text_test = vectorizer.transform(test_text.data)
    mnb = MultinomialNB(alpha=1.0)
    mnb.fit(X_text_train, train_text.target)
    text_pred = mnb.predict(X_text_test)
    print('20 Newsgroups accuracy:', accuracy_score(test_text.target, text_pred))
else:
    print('Transfer Challenge skipped. Set RUN_TRANSFER=True to run it.')

# Final — Model Reflection Card

## 1. Prior, likelihood và posterior khác nhau thế nào?
**[VIẾT]**

## 2. GaussianNB thực sự học gì từ tập train?
**[VIẾT]**

## 3. Trong case study `Needs_Support`, lỗi nào đáng chú ý hơn?
**[VIẾT]**

## 4. Vì sao `Exam_Score` không được dùng làm feature?
**[VIẾT]**

## 5. K-NN và GaussianNB khác nhau ở cách ra quyết định như thế nào?
**[VIẾT]**

## 6. Vì sao 20 Newsgroups dùng MultinomialNB thay vì GaussianNB?
**[VIẾT]**

## 7. Một câu chốt
> Dataset không đổi nhưng dự đoán có thể thay đổi vì...

**[VIẾT]**

## Trước khi push
```bash
python scripts/download_data.py
python tests/check_lab03.py
python -m pytest -q tests/test_nb_manual.py
git add .
git commit -m "Complete Lab 03 Naive Bayes"
git push
```